Multimodal RAG

In [1]:
#libraries
import fitz 
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

C:\Users\Dell\AppData\Local\Temp\ipykernel_11888\577449960.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

Loading/Initializing the CLIP Model

In [3]:
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,),

Embedding functions for image and text

In [4]:
### Embedding functions
def embed_image(image_data):
    """Embed image using CLIP"""
    if isinstance(image_data, str):  # If path
        image = Image.open(image_data).convert("RGB")
    else:  # If PIL Image
        image = image_data
    
    inputs=clip_processor(images=image,return_tensors="pt")
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
        # Normalize embeddings to unit vector
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()
    
def embed_text(text):
    """Embed text using CLIP."""
    inputs = clip_processor(
        text=text, 
        return_tensors="pt", 
        padding=True,
        truncation=True,
        max_length=77  # CLIP's max token length
    )
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)
        # Normalize embeddings
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()

Reading PDF

In [5]:
## Process PDF
pdf_path="multimodal_sample.pdf"
doc=fitz.open(pdf_path)
# Storage for all documents and embeddings
all_docs = []
all_embeddings = []
image_data_store = {}  # Store actual image data for LLM

# Text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
doc


Document('multimodal_sample.pdf')

Processing the Docs in the pdf

In [6]:
for i,page in enumerate(doc):
    ## process text
    text=page.get_text()
    if text.strip():
        ##create temporary document for splitting
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
        text_chunks = splitter.split_documents([temp_doc])

        #Embed each chunk using CLIP
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)



    ## process images
    ##Three Important Actions:

    ##Convert PDF image to PIL format
    ##Store as base64 for GPT-4V (which needs base64 images)
    ##Create CLIP embedding for retrieval

    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            
            # Convert to PIL Image
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            
            # Create unique identifier
            image_id = f"page_{i}_img_{img_index}"
            
            # Store image as base64 for later use with GPT-4V
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id] = img_base64
            
            # Embed image using CLIP
            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)
            
            # Create document for image
            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page": i, "type": "image", "image_id": image_id}
            )
            all_docs.append(image_doc)
            
        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

doc.close()


In [8]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
 Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')]

In [9]:
embeddings_array = np.array(all_embeddings)
data = zip(all_docs,embeddings_array)
data

Creating VectorStore

In [10]:
vectorstore = FAISS.from_embeddings(
    text_embeddings=[(doc.page_content,emb) for doc,emb in data],
    embedding=None,
    metadatas=[doc.metadata for doc in all_docs]
)
vectorstore

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


Initializing the LLM

In [35]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="qwen/qwen3.6-27b"
)

Fnc for querying vectorstore / retriever

In [36]:
def retrieve_multimodal(query,k=5):
    query_embedding = embed_text(query)
    results = vectorstore.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )
    return results

Fnc for formatting img and text before being sent to LLM

In [43]:
def create_multimodal_message(query,retrieved_docs):
    content = []
    content.append({
        "type":"text",
        "text": f"Question: {query}\n\nContext:\n"
    })

    #separate text and img docs
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]

    # add text context
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata['page']}]: {doc.page_content}"
            for doc in text_docs
        ])
        content.append({
            "type":"text",
            "text": f"Text excerpts:\n {text_context}"
        })
    
    # add images
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type":"text",
                "text":f"\n[Imagw from page {doc.metadata['page']}]:\n"
            })
            content.append({
                "type":"image_url",
                "image_url":{
                    "url": f"data:image/png;base64,{image_data_store[image_id]}"
                }
            })
    
    # adding instruction
    content.append({
        "type":"text",
        "text":"\n\n Please answer the question based on the provided text and images. IMPORTANT: PROVIDE THE ANSWER TO THE QUERY DIRECTLY NO EXPLANATION REGARDING WHAT THE USER IS ASKING IS NEEDED. ANSWER THE QUERY IN SHORT 3-4 LINES NOT MORE THAN THAT"
    })

    return HumanMessage(content=content)

RAG Pipeline

In [44]:
from pprint import pprint
def multimodal_rag_pipeline(query):
    context_docs = retrieve_multimodal(query,k=5)
    message = create_multimodal_message(query=query,retrieved_docs=context_docs)
    response = llm.invoke([message])

    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type = doc.metadata.get("type","unknown")
        page = doc.metadata.get("page","?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "...." if len(doc.page_content)>100 else doc.page_content
            print(f" - Text from page {page}: {preview}")
        else:
            print(f" - Image from page {page}")
        print("\n")

    return response.content

Testing out Multimodal RAG

In [45]:
queries = [
    "What does the chart on page 1 show about revenue trends?",
    "Summarize the main findings from the document",
    "What visual elements are present in the document?"
]
for query in queries:
    print(f"\nQuery: {query}")
    print("-"*50)
    answer = multimodal_rag_pipeline(query)
    print(f"Answer: {answer}")
    print("="*70)


Query: What does the chart on page 1 show about revenue trends?
--------------------------------------------------

Retrieved 2 documents:
 - Text from page 0: Annual Revenue Overview
This document summarizes the revenue trends across Q1, Q2, and Q3. As illust....


 - Image from page 0


Answer: 
<think>
The user wants to know about the revenue trends shown in the chart on page 1.
Looking at the provided context:
- The text is labeled "[Page 0]".
- The text says: "Annual Revenue Overview. This document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart below, revenue grew steadily with the highest growth recorded in Q3."
- The text further explains: "Q1 showed a moderate increase... Q2 outperformed Q1... Q3 had exponential growth..."
- The image provided is labeled "[Image from page 0]" and shows a bar chart with three bars (blue, green, red) increasing in height from left to right. This matches the description of Q1, Q2, and Q3 growth.

Wait, the questi